Rosbag

In [7]:
import cv2
import numpy as np
from pathlib import Path
from rosbags.highlevel import AnyReader
import csv

import os, sys

from tqdm import tqdm


In [8]:
rosbag_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/test/aracati/ARACATI_2017_8bits_full.bag'

In [9]:
# show content

with AnyReader([Path(rosbag_pth)]) as reader:
    counts = {}
    for connection, timestamp, rawdata in reader.messages():
        counts[connection.topic] = counts.get(connection.topic, 0) + 1
    
    for topic, count in counts.items():
        print(f"Topic: {topic} | Messages: {count}")


Topic: /pose_gt | Messages: 14436
Topic: /son/compressed | Messages: 14436
Topic: /dgps | Messages: 2641
Topic: /dgps_point | Messages: 2641
Topic: /cmd_vel | Messages: 14435
Topic: /surface/compressed | Messages: 7625
Topic: /usbl | Messages: 977
Topic: /usbl_point | Messages: 977


In [10]:
OUTPUT_DIR = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/aracati/data_example'
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(OUTPUT_DIR, "data.csv")
topic_sonar = '/son/compressed'
topic_pose = '/pose_gt'

with AnyReader([Path(rosbag_pth)]) as reader:
    # 1. Pobieramy połączenia
    connections = [x for x in reader.connections if x.topic in [topic_sonar, topic_pose]]
    
    # 2. Obliczamy całkowitą liczbę obrazów dla paska postępu
    # Szukamy połączenia dla sonaru, aby pobrać liczbę wiadomości
    sonar_conn = next((x for x in reader.connections if x.topic == topic_sonar), None)
    total_msgs = sonar_conn.msgcount if sonar_conn else 0

    with open(csv_path, 'w', newline='') as f, tqdm(total=total_msgs, desc="Przetwarzanie") as pbar:
        writer = csv.writer(f)
        writer.writerow(['step_idx', 'timestamp', 'pos_x', 'pos_y', 'pos_z', 
                         'quat_x', 'quat_y', 'quat_z', 'quat_w', 
                         'dvl_vel_x', 'dvl_vel_y', 'dvl_vel_z', 'dvl_alt', 
                         'imu_orient_x', 'imu_orient_y', 'imu_orient_z', 'imu_orient_w', 
                         'imu_gyro_x', 'imu_gyro_y', 'imu_gyro_z', 
                         'imu_accel_x', 'imu_accel_y', 'imu_accel_z'])
        
        current_pose = [0.0]*7 
        count = 0
        
        for connection, timestamp, rawdata in reader.messages(connections=connections):
            msg = reader.deserialize(rawdata, connection.msgtype)
            
            if connection.topic == topic_pose:
                # Jeśli msg to PoseStamped: msg.pose.position/orientation
                # Jeśli to Odometry, użyj: msg.pose.pose.position/orientation
                p = msg.pose.position
                o = msg.pose.orientation
                current_pose = [p.x, p.y, p.z, o.x, o.y, o.z, o.w]
            
            elif connection.topic == topic_sonar:
                encoded_img = np.frombuffer(msg.data, dtype=np.uint8)
                img = cv2.imdecode(encoded_img, cv2.IMREAD_COLOR)
                
                if img is not None:
                    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    filename = os.path.join(OUTPUT_DIR, f"{count}.png")
                    cv2.imwrite(filename, gray_img)
                    
                    # Zapis danych do CSV
                    row = [count, timestamp / 1e9] + current_pose + [0.0] * 15
                    writer.writerow(row)
                    
                    count += 1
                
                # Aktualizacja paska postępu przy każdym przetworzonym sonarze
                pbar.update(1)

print(f"\nGotowe! Dane zapisano w: {OUTPUT_DIR}")

Przetwarzanie: 100%|██████████| 14436/14436 [12:46<00:00, 18.83it/s]


Gotowe! Dane zapisano w: C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/aracati/data_example


In [ ]:
# OUTPUT_DIR = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/aracati/data_example'


# with AnyReader([Path(rosbag_pth)]) as reader:
#     # Znajdujemy połączenie dla Twojego topicu
#     connections = [x for x in reader.connections if x.topic == '/son/compressed']
    
#     if not connections:
#         print(f"Topic: {'/son/compressed'} is empty")
#         print("Available topics:")
#         for cn in reader.connections:
#             print(f" - {cn.topic} [{cn.msgtype}]")
#     else:
#         count = 0
#         for connection, timestamp, rawdata in reader.messages(connections=connections):
#             # Deserializacja

        
#             msg = reader.deserialize(rawdata, connection.msgtype)
#             encoded_img = np.frombuffer(msg.data, dtype=np.uint8)
#             img = cv2.imdecode(encoded_img, cv2.IMREAD_COLOR)

#             if img is not None:
#                 # Skoro to sonar, pewnie chcesz od razu skalę szarości:
#                 gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                
#                 # Zapis (używamy count dla nazwy)
#                 filename = f"{OUTPUT_DIR}/{count}.png"
#                 cv2.imwrite(str(filename), gray_img)
#             else:
#                 print(f"Błąd dekodowania obrazu przy count: {count}")
        
#         count += 1
#         if count % 50 == 0:
#             print(f"Processed {count} images...")

            
#         print(f"Ready! Exported files to dir: {OUTPUT_DIR}")

Processed 50 images...
Processed 100 images...
Processed 150 images...
Processed 200 images...
Processed 250 images...
Processed 300 images...
Processed 350 images...
Processed 400 images...
Processed 450 images...
Processed 500 images...
Processed 550 images...
Processed 600 images...
Processed 650 images...
Processed 700 images...
Processed 750 images...
Processed 800 images...
Processed 850 images...
Processed 900 images...
Processed 950 images...
Processed 1000 images...
Processed 1050 images...
Processed 1100 images...
Processed 1150 images...
Processed 1200 images...
Processed 1250 images...
Processed 1300 images...
Processed 1350 images...
Processed 1400 images...
Processed 1450 images...
Processed 1500 images...
Processed 1550 images...
Processed 1600 images...
Processed 1650 images...
Processed 1700 images...
Processed 1750 images...
Processed 1800 images...
Processed 1850 images...
Processed 1900 images...
Processed 1950 images...
Processed 2000 images...
Processed 2050 image